In [ ]:
import os, sys
import socket
from pathlib import Path
import numpy as np
import pandas as pd
import torch

np.random.seed(0)
torch.manual_seed(0)

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt

from vi_rnn.saving import load_model
from vi_rnn.datasets import SWM_dataset_multi
from fig_utils.decoding import extract_delay_act
from fig_utils.plots import (
    plot_cross_session_decoding_matrix,
    plot_cross_session_decoding_by_macaque,
)

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [ ]:
hostname = socket.gethostname()
print("hostname:", hostname)

# Update to your own path

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/macaque")
    path = "/home/matthijs/swm_rnn/data/"

else:
    out_dir = Path("/Users/matthijs/swm_rnn_cl/final_models/macaque")
    path = str(Path.cwd().parent / "data") + "/"


model_dirs = [
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_28_T_05_34_01",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_05_01_T_22_03_56",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_18_03_12",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_17_02_13",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_30_36",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_26_38",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_27_25",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_30_22",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_20_35",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_13_05",
]

In [ ]:
# --- controls ---

# --- data ---
n_trials_train = 150
n_trials_test = 60

# --- windows ---
time_decode = 10  # take 10 timebins (500 ms) before delay end

# --- run ---
run = False

In [ ]:
if run:
    rows = []

    for model_dir in model_dirs:
        model_dir = out_dir / Path(model_dir)
        name = model_dir.name

        vae, training_params, task_params = load_model(
            str(model_dir), load_encoder=False, backward_compat=False
        )
        task_params["path"] = path

        ll_final = float(training_params["ll"][-1])
        n_steps = int(len(training_params["ll"]))
        task = SWM_dataset_multi(task_params)
        (
            Z_inferred_sess_train,
            labels_sess_train,
            Z_inferred_sess_test,
            labels_sess_test,
        ) = extract_delay_act(
            vae, task, n_trials_train, n_trials_test, training_params, time_decode
        )

        rows.append(
            {
                "name": name,
                "path": str(model_dir),
                "ll_final": ll_final,
                "n_steps": n_steps,
                "macaque": task_params["sessions"][0][5:10],
                "Z_train": Z_inferred_sess_train,
                "labels_train": labels_sess_train,
                "Z_test": Z_inferred_sess_test,
                "labels_test": labels_sess_test,
            }
        )

In [ ]:
if run:
    df = pd.DataFrame(rows)
    # save
    df.to_pickle("../data/processed/df_cross_decoding_data.pkl")
else:
    df = pd.read_pickle("../data/processed/df_cross_decoding_data.pkl")

In [ ]:
n_sessions = 15
n_pos = 3

# Count animals
n_groot = sum(df["macaque"][s] == "groot" for s in range(len(df)))
n_ocean = sum(df["macaque"][s] == "ocean" for s in range(len(df)))

# Output arrays
acc_groot = np.zeros((n_sessions, n_sessions, n_pos, n_groot))
acc_ocean = np.zeros((n_sessions, n_sessions, n_pos, n_ocean))

groot_index = -1
ocean_index = -1

print("Starting decoding...")

for seed in range(len(df)):

    macaque = df["macaque"][seed]

    if macaque == "groot":
        groot_index += 1
        out_acc = acc_groot
        animal_idx = groot_index
    else:
        ocean_index += 1
        out_acc = acc_ocean
        animal_idx = ocean_index

    print(f"\nSeed {seed} | macaque = {macaque}")

    Z_inferred_sess_train = df["Z_train"][seed]
    Z_inferred_sess_test = df["Z_test"][seed]
    labels_sess_train = df["labels_train"][seed]
    labels_sess_test = df["labels_test"][seed]
    for p in range(n_pos):

        print(f"  Position {p}")

        for i in range(n_sessions):
            X_train = Z_inferred_sess_train[i, :]
            y_train = labels_sess_train[i, :, p]

            clf = SVC(kernel="linear")
            clf.fit(X_train, y_train)

            for j in range(n_sessions):

                X_test = Z_inferred_sess_test[j, :]
                y_test = labels_sess_test[j, :, p]

                y_pred = clf.predict(X_test)

                acc = accuracy_score(y_test, y_pred)

                out_acc[i, j, p, animal_idx] = acc

In [ ]:
print("\n================ GROOT SUMMARY ================")

for p in range(n_pos):

    print(f"\nPosition {p}")

    for i in range(n_sessions):

        vals = acc_groot[i, :, p, :].flatten()

        print(
            f"Train session {i:02d} | "
            f"mean={vals.mean():.3f} "
            f"std={vals.std():.3f}"
        )

print("\n================ OCEAN SUMMARY ================")

for p in range(n_pos):

    print(f"\nPosition {p}")

    for i in range(n_sessions):

        vals = acc_ocean[i, :, p, :].flatten()

        print(
            f"Train session {i:02d} | "
            f"mean={vals.mean():.3f} "
            f"std={vals.std():.3f}"
        )

In [ ]:
acc_means = (np.mean(acc_ocean, axis=(2, 3)) + np.mean(acc_groot, axis=(2, 3))) / 2

In [ ]:
plot_cross_session_decoding_by_macaque(
    acc_groot,
    acc_ocean,
    reduce_axes=(2, 3),
    box_w=0.6,
    box_h=0.6,
    dpi=300,
    save_path="../paper_figures/fig2_3.pdf",
    show=True,
    norm_vmin=0.75,
    cbar_ticks=(0.75, 1.0),
    panel_gap_x=0.15,
)

In [ ]:
print(acc_ocean.mean(axis=(0, 1, 3)))
print(acc_ocean.mean(axis=(0, 1)).std(axis=(1)))

In [ ]:
print(acc_groot.mean(axis=(0, 1, 3)))
print(acc_groot.mean(axis=(0, 1)).std(axis=(1)))